In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

from collections import defaultdict
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys.
    """
    rng = np.random.default_rng(seed)
    transactions: list[set[str]] = []
    for _ in range(n):
        basket: set[str] = set()
        for bundle_items, prob in BUNDLES:
            if rng.random() < prob:
                for item in bundle_items:
                    if rng.random() < 0.85:
                        basket.add(item)
        n_random = rng.poisson(2)
        if n_random > 0:
            random_items = rng.choice(
                PRODUCTS, size=int(min(n_random, 5)), replace=False
            )
            basket.update(random_items)
        if basket:
            transactions.append(basket)
    return transactions


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. Used as input to mlxtend FP-Growth (via .to_pandas()).
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.4: Rule-Based Features for Supervised Classification
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Engineer features from discovered association rules
#   - Compare a product-presence baseline against a rule-enhanced model
#   - Measure whether explicit rules add signal over raw one-hot features
#   - Attribute model importance across product vs rule feature groups
#   - See the forward connection to matrix factorisation and neural nets
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py
#   - 03_rule_evaluation.py
#   - MLFP03 Exercise 1 (feature engineering)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — rules as handcrafted co-occurrence features
#   2. Build — turn actionable rules into numeric feature columns
#   3. Train — logistic regression + random forest, baseline vs combined
#   4. Visualise — metric comparison + feature importance attribution
#   5. Apply — NTUC Link high-value shopper scoring for CRM
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import combinations

import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



## THEORY — Rules as Handcrafted Co-Occurrence Features


In [ ]:
# A linear model trained only on product-presence one-hot features sees
# each product as independent. It cannot represent "this basket contains
# the breakfast bundle" as a single signal — at best it learns a weight
# per product and adds them up.
#
# Association rules give you that missing structure for free. Every
# actionable rule X -> Y becomes one or two new binary features:
#
#   ant_present[r]   = 1 if the basket contains X
#   full_present[r]  = 1 if the basket contains X AND Y
#
# Plus a handful of aggregates:
#
#   n_rules_triggered  = how many rules fire on this basket
#   total_rule_lift    = sum of lifts of the rules that fire
#   max_rule_lift      = strongest rule that fires
#
# This is the MANUAL version of what matrix factorisation (Ex 7) and
# neural networks (Ex 8) will do AUTOMATICALLY. The progression is:
#
#   Manual rules  ->  Linear factorisation  ->  Nonlinear neural nets
#   explicit         compressed latent         learned non-linearity
#   interpretable    partially interpretable   least interpretable
#   cheap            medium                    expensive
#
# Ex 5.4 is the bottom rung. Every step above this file is a different
# way to discover co-occurrence structure without hand-writing rules.



## MINI-APRIORI + RULE GENERATION (inline, so this file stands alone)


Inline Apriori — same contract as 01_apriori_from_scratch.apriori().


In [ ]:
def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


def _rules_from_itemsets(
    freq: dict[frozenset[str], float],
    min_confidence: float = 0.4,
    min_lift: float = 1.5,
) -> list[dict]:
    rules: list[dict] = []
    for itemset, support in freq.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ant_tuple in combinations(items, r):
                antecedent = frozenset(ant_tuple)
                consequent = itemset - antecedent
                supp_ant = freq.get(antecedent)
                supp_con = freq.get(consequent)
                if supp_ant is None or supp_con is None:
                    continue
                confidence = support / supp_ant
                lift = confidence / supp_con
                if confidence >= min_confidence and lift > min_lift:
                    rules.append(
                        {
                            "antecedent": antecedent,
                            "consequent": consequent,
                            "support": support,
                            "confidence": confidence,
                            "lift": lift,
                        }
                    )
    rules.sort(key=lambda r: -r["lift"])
    return rules



## TASK 2 — BUILD: rule-based feature engineer


Columns produced:
      - ``rule{i}_antecedent`` — 1 if the antecedent is present
      - ``rule{i}_full``       — 1 if antecedent AND consequent are present
      - ``n_rules_triggered``  — count of fully-matched rules
      - ``total_rule_lift_x100`` — sum of matched rule lifts (scaled int)
      - ``max_rule_lift_x100``   — max matched rule lift (scaled int)


In [ ]:
def engineer_rule_features(
    transactions: list[set[str]],
    rules: list[dict],
) -> pl.DataFrame:
    rows: list[dict[str, int]] = []
    for txn in transactions:
        txn_set = frozenset(txn)
        row: dict[str, int] = {}
        total_lift = 0.0
        n_triggered = 0
        max_lift = 0.0

        for idx, rule in enumerate(rules):
            ant_present = int(rule["antecedent"].issubset(txn_set))
            full_present = int(
                (rule["antecedent"] | rule["consequent"]).issubset(txn_set)
            )
            row[f"rule{idx}_antecedent"] = ant_present
            row[f"rule{idx}_full"] = full_present
            if full_present:
                n_triggered += 1
                total_lift += float(rule["lift"])
                if float(rule["lift"]) > max_lift:
                    max_lift = float(rule["lift"])

        row["n_rules_triggered"] = n_triggered
        row["total_rule_lift_x100"] = int(total_lift * 100)
        row["max_rule_lift_x100"] = int(max_lift * 100)
        rows.append(row)

    return pl.DataFrame(rows).fill_null(0)



## TASK 3 — TRAIN: baseline vs combined classifiers


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

# Target = "high-value shopper" (basket size >= 6 items).
basket_sizes = np.array([len(t) for t in transactions])
HIGH_VALUE_THRESHOLD = 6
y = (basket_sizes >= HIGH_VALUE_THRESHOLD).astype(int)
print(f"\n  Target: high-value shopper (basket >= {HIGH_VALUE_THRESHOLD} items)")
print(f"  Positive rate: {y.mean():.1%}")

# --- Baseline features: raw product presence ---
onehot = transactions_to_onehot(transactions)
X_baseline = onehot.to_numpy().astype(np.float64)
print(f"\n  Baseline features (product presence): {X_baseline.shape[1]}")

# --- Mine rules and engineer features ---
print("\n=== Mining rules for feature engineering ===")
freq = _apriori(transactions, min_support=0.03)
rules = _rules_from_itemsets(freq, min_confidence=0.4, min_lift=1.5)
top_rules = rules[:20]
print(f"  Actionable rules found: {len(rules)}")
print(f"  Using top {len(top_rules)} rules as features")

rule_df = engineer_rule_features(transactions, top_rules)
X_rules = rule_df.to_numpy().astype(np.float64)
X_combined = np.hstack([X_baseline, X_rules])
print(f"  Rule features:     {X_rules.shape[1]}")
print(f"  Combined features: {X_combined.shape[1]}")


def _split_and_scale(X: np.ndarray, y: np.ndarray, scale: bool):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    if scale:
        s = StandardScaler()
        X_tr = s.fit_transform(X_tr)
        X_te = s.transform(X_te)
    return X_tr, X_te, y_tr, y_te


results: dict[str, dict[str, float]] = {}

print("\n=== Model: Logistic Regression ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=True)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_te)
    y_proba = lr.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"LR: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")

print("\n=== Model: Random Forest ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=False)
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    y_proba = rf.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"RF: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")



### Checkpoint


In [ ]:
assert (
    X_combined.shape[1] > X_baseline.shape[1]
), "Combined should add columns to the baseline, not replace them"
assert X_rules.shape[1] > 0, "Should have produced at least one rule feature"
lr_baseline = results["LR: Baseline"]["auc_roc"]
lr_combined = results["LR: Combined"]["auc_roc"]
rf_baseline = results["RF: Baseline"]["auc_roc"]
assert lr_baseline > 0.5, "Baseline LR should beat random"
assert rf_baseline > 0.5, "Baseline RF should beat random"
assert (
    lr_combined >= lr_baseline - 0.05
), "Adding rule features should not significantly regress LR"
print("\n[ok] Checkpoint passed — rule-enhanced model trained + compared\n")



## TASK 4 — VISUALISE: feature importance attribution


In [ ]:
X_tr, X_te, y_tr, y_te = _split_and_scale(X_combined, y, scale=False)
rf_combined = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf_combined.fit(X_tr, y_tr)
importances = rf_combined.feature_importances_

product_importance = float(importances[: X_baseline.shape[1]].sum())
rule_importance = float(importances[X_baseline.shape[1] :].sum())
total = product_importance + rule_importance

print("=== Feature Importance Attribution (RF combined) ===")
print(f"  Product features contribute: {product_importance / total:.1%}")
print(f"  Rule features contribute:    {rule_importance / total:.1%}")

all_feature_names = list(onehot.columns) + list(rule_df.columns)
top_idx = np.argsort(importances)[::-1][:15]
print("\n  Top 15 features in the combined model:")
for idx in top_idx:
    fname = all_feature_names[idx]
    ftype = "product" if idx < X_baseline.shape[1] else "rule"
    print(f"    [{ftype:>7}] {fname:<40} {importances[idx]:.4f}")

# Metric comparison frame (for notebooks to chart)
metric_rows = [{"model": k, **v} for k, v in results.items()]
metric_df = pl.DataFrame(metric_rows)
metric_df.write_csv(OUTPUT_DIR / "rule_features_metrics.csv")
print(f"\n  Saved: {OUTPUT_DIR / 'rule_features_metrics.csv'}")



### Visualisation


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# (A) Feature importance: product vs rule groups (pie + top-15 bar)
fig_imp = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "pie"}, {"type": "xy"}]],
    subplot_titles=["Importance by Group", "Top 15 Features"],
)
fig_imp.add_trace(
    go.Pie(
        labels=["Product Features", "Rule Features"],
        values=[product_importance, rule_importance],
        marker_colors=["#636EFA", "#EF553B"],
        textinfo="label+percent",
    ),
    row=1,
    col=1,
)
top_names = [all_feature_names[i] for i in top_idx]
top_vals = [float(importances[i]) for i in top_idx]
top_colors = ["#636EFA" if idx < X_baseline.shape[1] else "#EF553B" for idx in top_idx]
fig_imp.add_trace(
    go.Bar(
        x=top_vals,
        y=top_names,
        orientation="h",
        marker_color=top_colors,
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_imp.update_layout(
    title="Feature Importance: Product (blue) vs Rule (red) Features",
    height=500,
    width=1000,
)
fig_imp.update_yaxes(autorange="reversed", row=1, col=2)
imp_path = OUTPUT_DIR / "04_feature_importance.html"
fig_imp.write_html(str(imp_path))
print(f"[viz] Feature importance: {imp_path}")

# (B) Accuracy improvement: baseline vs combined across models
model_names = list(results.keys())
acc_vals = [results[k]["accuracy"] for k in model_names]
auc_vals = [results[k]["auc_roc"] for k in model_names]
fig_acc = go.Figure()
fig_acc.add_trace(
    go.Bar(
        x=model_names,
        y=acc_vals,
        name="Accuracy",
        marker_color="#636EFA",
        text=[f"{v:.3f}" for v in acc_vals],
        textposition="outside",
    )
)
fig_acc.add_trace(
    go.Bar(
        x=model_names,
        y=auc_vals,
        name="AUC-ROC",
        marker_color="#EF553B",
        text=[f"{v:.3f}" for v in auc_vals],
        textposition="outside",
    )
)
fig_acc.update_layout(
    title="Model Comparison: Baseline vs Rules-Only vs Combined",
    xaxis_title="Model Variant",
    yaxis_title="Score",
    barmode="group",
    yaxis_range=[0, 1.1],
)
acc_path = OUTPUT_DIR / "04_accuracy_comparison.html"
fig_acc.write_html(str(acc_path))
print(f"[viz] Accuracy comparison: {acc_path}")

# INTERPRETATION: For a simple linear model, rule features typically add
# 2-5 points of AUC because the LR cannot represent "all three breakfast
# items present" without an explicit interaction. For a random forest,
# the lift is smaller (sometimes zero) because the tree already learns
# interactions implicitly via branching. This is the exact reason Ex 7
# (matrix factorisation) and Ex 8 (neural nets) exist — they learn the
# same co-occurrence structure WITHOUT requiring you to pre-specify rules.



## TASK 5 — APPLY: NTUC Link high-value shopper scoring for CRM


In [ ]:
# SCENARIO: NTUC Link is Singapore's largest grocery loyalty programme
# (~1.7M members). The CRM team wants a model that scores each basket
# right after checkout, flagging the top 20% of baskets as "high value"
# for targeted next-trip offers (e.g., free delivery, priority slots).
#
# Two constraints drive the design:
#   - The model must be AUDITABLE (regulator + internal governance)
#   - The feature engineering must be REPRODUCIBLE (monthly re-training)
#
# Rule-based features meet both. Every feature column has a business
# name you can trace to a specific association rule, and the mining
# pipeline reruns monthly against the previous 90 days of baskets.
# Matrix factorisation (Ex 7) would score slightly higher on AUC but
# loses the per-feature interpretability that the CRM team needs when
# explaining a segment to senior management.
#
# BUSINESS IMPACT: A 3-5% improvement in high-value precision at the
# top 20% cutoff translates to roughly 25,000-42,000 members receiving
# targeted offers who would otherwise have been missed. Typical offer
# ROI at that scale is S$4-6 per activated member — ~S$100K-250K/month
# in incremental campaign contribution. On top of that, the rule-based
# column names feed directly into the "why did this shopper qualify?"
# explainability panel the CRM team shows regulators.
#
# LIMITATIONS:
#   - Rules captured here are support >= 3%, which is about the floor
#     for a 1.7M-member base; rarer categories (baby care, pet food)
#     need their own mining run with lower thresholds
#   - Target is basket size, a proxy for value — for true $-value
#     segmentation, weight by SKU price (out of scope here)



## REFLECTION


[x] Engineered numeric features from discovered association rules
  [x] Compared baseline product-presence vs rule-enhanced models
  [x] Attributed feature importance across product and rule groups
  [x] Identified a production scenario (NTUC Link CRM scoring) where
      explicit rule features are preferred over learned embeddings

  KEY INSIGHT: Association rules are the MANUAL end of a spectrum that
  runs all the way to deep learning.

    Manual rules  ->  Linear factorisation  ->  Nonlinear neural nets
    (this file)      (Ex 7)                    (Ex 8)

  Every technique to the right of this one discovers the same
  co-occurrence structure, just with less human steering and less
  interpretability.

  Next: Exercise 6 moves to UNSTRUCTURED text — TF-IDF from scratch,
  NMF topic modelling, and topic quality evaluation via NPMI coherence.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

